In [ ]:
!git clone https://github.com/aranggitoar/conn_intrp.git
%cd conn_intrp
!pip install -e . -qq

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/Kuliah/tesis/docVQA"
WEIGHTS_PATH = "/content/drive/MyDrive/Kuliah/tesis/weights"
OUTPUTS_PATH = "/content/drive/MyDrive/Kuliah/tesis/outputs"

In [ ]:
from pathlib import Path

from conn_intrp import compute_category_means, load_docvqa, run_ablation
from conn_intrp.models import InternVLAdapter

adapter = InternVLAdapter("OpenGVLab/InternVL3_5-2B-HF")

image_base_path = Path(DATASET_PATH)
_, data_categorized = load_docvqa(image_base_path / "train_v1.0_withQT.json")

# From Phase 1 — replace with your filtered directions
directions_to_ablate = [23, 70, 255]
batch_size = 1

run_dir = Path(OUTPUTS_PATH) / "ablation_internvl3_5_2b"
run_dir.mkdir(parents=True, exist_ok=True)

print(f"S: {adapter.S.shape}, patches: {adapter.n_patches}, categories: {len(data_categorized)}")

In [ ]:
per_category_coefficients, per_category_a_star, global_a_star = compute_category_means(
    adapter, data_categorized,
    batch_size=batch_size, image_base_path=image_base_path, run_dir=run_dir,
)
print("Global mean computed")

In [ ]:
run_ablation(
    adapter, data_categorized,
    per_category_coefficients, per_category_a_star, global_a_star,
    directions_to_ablate=directions_to_ablate,
    batch_size=batch_size, K=15,
    image_base_path=image_base_path, run_dir=run_dir,
)